# TB-CXR: Severity-Stratified Tuberculosis Classification for Immigration Medical Examination
## Deep Transfer Learning with DenseNet-121, GradCAM, and Uncertainty Quantification

**Author:** Azizur Rahman  
**Affiliation:** Indiana Wesleyan University · RadTH Technologies  
**Paper:** *Severity-Stratified Tuberculosis Classification from Chest Radiographs Aligned with CDC Immigration Medical Examination Guidelines*  
**Target Venue:** Journal of Biomedical Informatics

---
### Notebook Contents
| # | Section |
|---|--------|
| 1 | Environment Setup & Imports |
| 2 | Dataset Loading & 4-Class Label Preparation |
| 3 | Data Augmentation & PyTorch DataLoader |
| 4 | Model Architecture (DenseNet-121 + ResNet-50) |
| 5 | Asymmetric Triage Loss Function |
| 6 | Training with Transfer Learning |
| 7 | Cross-Dataset Evaluation (Montgomery + Shenzhen) |
| 8 | GradCAM Explainability |
| 9 | Uncertainty Quantification (Monte Carlo Dropout) |
| 10 | Demographic Equity Analysis (CTEI) |
| 11 | Results Summary & Paper Tables |

## 1. Environment Setup & Imports

In [ ]:
!pip install torch torchvision timm
!pip install scikit-learn pandas numpy matplotlib seaborn tqdm
!pip install grad-cam opencv-python pillow
!pip install kaggle

In [ ]:
import os, random, warnings, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, f1_score, roc_auc_score,
    confusion_matrix, precision_score, recall_score
)
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

NUM_CLASSES = 4
LABEL_NAMES  = ['Normal', 'Inactive_TB', 'Active_TB', 'Severe_TB']
USCIS_LABELS = ['Cleared', 'Class B2', 'Class B1', 'Class A']
IMG_SIZE     = 224
BATCH_SIZE   = 32

print(f'Device : {DEVICE}')
print(f'PyTorch: {torch.__version__}')

## 2. Dataset Loading & 4-Class Label Preparation

### Download Instructions
```bash
# Option 1 — Kaggle CLI (recommended)
kaggle datasets download -d tawsifurrahman/tuberculosis-tb-chest-xray-dataset

# Option 2 — TBX11K
kaggle datasets download -d usmanshams/tbx-11

# Option 3 — Montgomery + Shenzhen combined
kaggle datasets download -d raddar/tuberculosis-chest-xrays-montgomery
kaggle datasets download -d raddar/tuberculosis-chest-xrays-shenzhen
```

### 4-Class CDC/USCIS Severity Mapping
| Class | Label | X-Ray Finding | USCIS Outcome |
|---|---|---|---|
| 0 | Normal | Clear lungs | Cleared |
| 1 | Inactive TB | Calcified nodules, fibrous scars | Class B2 — Cleared with follow-up |
| 2 | Active TB | Infiltrates, consolidation (no cavity) | Class B1 — Further evaluation |
| 3 | Severe TB | Cavitation, miliary pattern | Class A — Not cleared |

In [ ]:
# ── Keyword-based severity grading from radiology reports ──────────────────
SEVERE_KEYWORDS   = ['cavit','miliary','bilateral extensive','large effusion',
                     'extensive consolidation','disseminated','progressive']
ACTIVE_KEYWORDS   = ['infiltrat','consolidat','ground glass','opacity',
                     'airspace','active','pneumonic','lesion','exudate']
INACTIVE_KEYWORDS = ['calcif','fibro','scar','healed','old','chronic',
                     'nodule','granuloma','pleural thicken','residual']

def classify_severity(report: str, has_tb: bool) -> int:
    if not has_tb:
        return 0
    if not isinstance(report, str) or len(report.strip()) == 0:
        return 2  # Default TB-positive to Active if no report
    text = report.lower()
    if any(k in text for k in SEVERE_KEYWORDS):   return 3
    if any(k in text for k in ACTIVE_KEYWORDS):   return 2
    if any(k in text for k in INACTIVE_KEYWORDS): return 1
    return 2


def load_dataset(data_dir: str, dataset_name: str = 'combined') -> pd.DataFrame:
    """
    Load chest X-ray images and assign 4-class severity labels.
    Expects folder structure:
        data_dir/
            Normal/     (or TB-Negative/)
            TB/         (or TB-Positive/)
    OR flat folder with filename-based labels (Shenzhen: *_0.png=normal, *_1.png=TB)
    """
    rows = []

    # Try structured folder approach
    for folder, has_tb in [('Normal', False), ('TB-Negative', False),
                            ('TB', True),      ('TB-Positive', True)]:
        folder_path = os.path.join(data_dir, folder)
        if not os.path.exists(folder_path):
            continue
        report_dir = os.path.join(data_dir, 'ClinicalReadings')
        for fname in os.listdir(folder_path):
            if not fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                continue
            img_path = os.path.join(folder_path, fname)
            report = ''
            rpath  = os.path.join(report_dir, fname.replace('.png','.txt'))
            if os.path.exists(rpath):
                with open(rpath, 'r', errors='ignore') as f:
                    report = f.read()
            label = classify_severity(report, has_tb)
            rows.append({'image_path': img_path, 'label': label,
                         'label_name': LABEL_NAMES[label],
                         'dataset': dataset_name})

    if not rows:
        # Flat folder with filename-based labels (Shenzhen convention)
        for fname in os.listdir(data_dir):
            if not fname.lower().endswith(('.png', '.jpg')):
                continue
            has_tb = fname.split('_')[-1].replace('.png','') == '1'
            label  = classify_severity('', has_tb)
            rows.append({'image_path': os.path.join(data_dir, fname),
                         'label': label,
                         'label_name': LABEL_NAMES[label],
                         'dataset': dataset_name})

    df = pd.DataFrame(rows)
    print(f'{dataset_name}: {len(df)} images')
    print(df['label_name'].value_counts().to_string())
    return df


# ── Load all datasets ───────────────────────────────────────────────────────
DATA_ROOT = 'data'  # Change to your actual data directory

# Uncomment and adjust paths when running:
# df_main  = load_dataset(f'{DATA_ROOT}/montgomery',  'Montgomery')
# df_valid = load_dataset(f'{DATA_ROOT}/shenzhen',    'Shenzhen')
# df_tbx   = load_dataset(f'{DATA_ROOT}/tbx11k',      'TBX11K')

# For demo — create synthetic DataFrame structure
print('Dataset loading functions ready.')
print('Update DATA_ROOT and uncomment load_dataset calls after downloading data.')

## 3. Data Augmentation & PyTorch DataLoader

In [ ]:
# ── Transforms ─────────────────────────────────────────────────────────────
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


class TBDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['image_path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(row['label'], dtype=torch.long)


def make_loaders(train_df, val_df, test_df):
    # Weighted sampler to handle class imbalance
    class_counts  = train_df['label'].value_counts().sort_index().values
    class_weights = 1.0 / class_counts
    sample_weights = class_weights[train_df['label'].values]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

    train_ldr = DataLoader(TBDataset(train_df, train_transforms),
                           batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
    val_ldr   = DataLoader(TBDataset(val_df,   val_transforms),
                           batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_ldr  = DataLoader(TBDataset(test_df,  val_transforms),
                           batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    return train_ldr, val_ldr, test_ldr

print('DataLoader functions ready.')

## 4. Model Architecture

In [ ]:
def build_densenet121(num_classes=NUM_CLASSES, pretrained=True, dropout=0.5):
    model = models.densenet121(weights='DEFAULT' if pretrained else None)
    in_features = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, num_classes)
    )
    return model


def build_resnet50(num_classes=NUM_CLASSES, pretrained=True, dropout=0.5):
    model = models.resnet50(weights='DEFAULT' if pretrained else None)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, num_classes)
    )
    return model


# Test architecture
dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE)
dn121 = build_densenet121()
rn50  = build_resnet50()
print(f'DenseNet-121 output: {dn121(dummy).shape}')
print(f'ResNet-50    output: {rn50(dummy).shape}')

total_dn  = sum(p.numel() for p in dn121.parameters())
total_rn  = sum(p.numel() for p in rn50.parameters())
print(f'DenseNet-121 params: {total_dn:,}')
print(f'ResNet-50    params: {total_rn:,}')

## 5. Asymmetric Triage Loss Function

Missing a Severe TB case (Class A) is far more dangerous than over-triaging a Normal case.
We penalize misclassification of Severe TB 5× more than Normal.

In [ ]:
class AsymmetricTriageLoss(nn.Module):
    """
    Weighted cross-entropy with asymmetric clinical penalties.
    Higher weight for Severe TB (Class A) misclassification.

    Weights reflect clinical cost:
        Normal      = 1.0  (low cost to miss)
        Inactive TB = 2.0  (moderate — needs follow-up)
        Active TB   = 3.0  (high — infectious risk)
        Severe TB   = 5.0  (critical — Class A, must not miss)
    """
    def __init__(self, weights=None, device=DEVICE):
        super().__init__()
        if weights is None:
            weights = torch.tensor([1.0, 2.0, 3.0, 5.0], dtype=torch.float32)
        self.ce = nn.CrossEntropyLoss(weight=weights.to(device))

    def forward(self, logits, targets):
        return self.ce(logits, targets)


# Compare standard vs asymmetric loss
print('Loss function: AsymmetricTriageLoss')
print('Class weights:')
for i, (name, w) in enumerate(zip(LABEL_NAMES, [1.0, 2.0, 3.0, 5.0])):
    print(f'  {i} - {name:<15} weight={w}')

## 6. Training with Transfer Learning

In [ ]:
from transformers import get_linear_schedule_with_warmup

def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, preds_all, trues_all = 0, [], []
    for imgs, labels in tqdm(loader, desc='Train', leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if scheduler: scheduler.step()
        total_loss += loss.item()
        preds_all.extend(logits.argmax(1).cpu().numpy())
        trues_all.extend(labels.cpu().numpy())
    f1 = f1_score(trues_all, preds_all, average='macro', zero_division=0)
    return total_loss / len(loader), f1


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, preds_all, trues_all = 0, [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        total_loss += criterion(logits, labels).item()
        preds_all.extend(logits.argmax(1).cpu().numpy())
        trues_all.extend(labels.cpu().numpy())
    f1 = f1_score(trues_all, preds_all, average='macro', zero_division=0)
    return f1, total_loss / len(loader), preds_all, trues_all


def run_training(model, train_ldr, val_ldr, model_name,
                 epochs=15, lr=2e-4, ckpt_dir='checkpoints'):
    ckpt_path = os.path.join(ckpt_dir, model_name, 'best.pt')
    os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)

    if os.path.exists(ckpt_path):
        print(f'Checkpoint found — loading {model_name}')
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        return model, {'train_loss':[], 'val_loss':[], 'train_f1':[], 'val_f1':[]}

    model = model.to(DEVICE)
    criterion = AsymmetricTriageLoss()
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    total_steps = len(train_ldr) * epochs
    scheduler   = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps
    )

    best_f1, best_state = 0, None
    history = {'train_loss':[], 'val_loss':[], 'train_f1':[], 'val_f1':[]}

    print(f'\nTraining {model_name} for {epochs} epochs...')
    print('=' * 65)
    for epoch in range(1, epochs + 1):
        tl, tf = train_epoch(model, train_ldr, optimizer, scheduler, criterion, DEVICE)
        vf, vl, _, _ = evaluate(model, val_ldr, criterion, DEVICE)
        history['train_loss'].append(tl); history['val_loss'].append(vl)
        history['train_f1'].append(tf);  history['val_f1'].append(vf)
        print(f'Epoch {epoch:>2}/{epochs}  '
              f'train_loss={tl:.4f}  train_F1={tf:.4f}  '
              f'val_loss={vl:.4f}  val_F1={vf:.4f}')
        if vf > best_f1:
            best_f1   = vf
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    torch.save(best_state, ckpt_path)
    print(f'Best val F1: {best_f1:.4f} | Saved to {ckpt_path}')
    return model, history


print('Training utilities ready.')
print('Run after loading dataset:')
print('  train_ldr, val_ldr, test_ldr = make_loaders(train_df, val_df, test_df)')
print('  dn121, hist_dn = run_training(build_densenet121(), train_ldr, val_ldr, "densenet121")')
print('  rn50,  hist_rn = run_training(build_resnet50(),   train_ldr, val_ldr, "resnet50")')

## 6b. Training Curves

In [ ]:
def plot_training_curves(hist_dn, hist_rn):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(hist_dn['train_loss'], label='DenseNet train', color='#2980B9', linestyle='--')
    ax1.plot(hist_dn['val_loss'],   label='DenseNet val',   color='#2980B9')
    ax1.plot(hist_rn['train_loss'], label='ResNet-50 train',color='#E74C3C', linestyle='--')
    ax1.plot(hist_rn['val_loss'],   label='ResNet-50 val',  color='#E74C3C')
    ax1.set(xlabel='Epoch', ylabel='Asymmetric Triage Loss', title='Training & Validation Loss')
    ax1.legend(); ax1.grid(alpha=0.3)

    ax2.plot(hist_dn['val_f1'], label='DenseNet-121', color='#2980B9', marker='o')
    ax2.plot(hist_rn['val_f1'], label='ResNet-50',    color='#E74C3C', marker='s')
    ax2.set(xlabel='Epoch', ylabel='Macro F1', title='Validation Macro F1')
    ax2.legend(); ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

# plot_training_curves(hist_dn, hist_rn)  # Uncomment after training

## 7. Cross-Dataset Evaluation (Montgomery + Shenzhen external validation)

In [ ]:
def compute_ctei(f1_dict: dict) -> float:
    """Cross-Demographic Triage Equity Index (adapted from MiST CTEI)."""
    vals = np.array(list(f1_dict.values()))
    mu, sigma = vals.mean(), vals.std()
    return float(1 - sigma / mu) if mu > 0 else 0.0


def evaluate_per_class(model, loader, device):
    model.eval()
    preds_all, trues_all = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            logits = model(imgs.to(device))
            preds_all.extend(logits.argmax(1).cpu().numpy())
            trues_all.extend(labels.numpy())

    macro_f1 = f1_score(trues_all, preds_all, average='macro', zero_division=0)
    report   = classification_report(trues_all, preds_all,
                                     target_names=LABEL_NAMES, zero_division=0)
    per_class_f1 = f1_score(trues_all, preds_all, average=None, zero_division=0)
    severe_recall = recall_score(
        [1 if t == 3 else 0 for t in trues_all],
        [1 if p == 3 else 0 for p in preds_all],
        zero_division=0
    )
    return {
        'macro_f1': macro_f1,
        'per_class_f1': per_class_f1,
        'severe_recall': severe_recall,
        'report': report,
        'preds': preds_all,
        'trues': trues_all
    }


def cross_dataset_evaluation(model, external_df, model_name):
    """Evaluate on held-out external dataset (cross-dataset generalization)."""
    ext_ldr = DataLoader(TBDataset(external_df, val_transforms),
                         batch_size=BATCH_SIZE, shuffle=False)
    results = evaluate_per_class(model, ext_ldr, DEVICE)
    print(f'\n{"="*60}')
    print(f'External Validation — {model_name}')
    print(f'{"="*60}')
    print(results['report'])
    print(f'Severe TB Recall (Class A): {results["severe_recall"]:.4f}')
    return results


print('Evaluation functions ready.')

## 8. GradCAM Explainability
Highlights which lung regions drove each severity classification.

In [ ]:
import cv2

class GradCAM:
    def __init__(self, model, target_layer):
        self.model        = model
        self.target_layer = target_layer
        self.gradients    = None
        self.activations  = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def generate(self, img_tensor, class_idx=None):
        self.model.eval()
        img_tensor = img_tensor.unsqueeze(0).to(DEVICE)
        logits = self.model(img_tensor)
        if class_idx is None:
            class_idx = logits.argmax().item()
        self.model.zero_grad()
        logits[0, class_idx].backward()
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam     = (weights * self.activations).sum(dim=1, keepdim=True)
        cam     = F.relu(cam)
        cam     = cam.squeeze().cpu().numpy()
        cam     = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        cam     = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
        return cam, class_idx


def visualize_gradcam(model, image_paths, titles=None):
    target_layer = model.features.denseblock4.denselayer16.conv2
    gradcam = GradCAM(model, target_layer)

    fig, axes = plt.subplots(2, len(image_paths), figsize=(4 * len(image_paths), 8))
    if len(image_paths) == 1:
        axes = axes.reshape(2, 1)

    inv_normalize = transforms.Normalize(
        mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
        std=[1/0.229, 1/0.224, 1/0.225]
    )

    for i, img_path in enumerate(image_paths):
        img_raw = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
        img_t   = val_transforms(img_raw)
        cam, pred_class = gradcam.generate(img_t)

        img_np  = np.array(inv_normalize(img_t).permute(1,2,0).clamp(0,1))
        heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0
        overlay = 0.6 * img_np + 0.4 * heatmap

        axes[0, i].imshow(img_np)
        axes[0, i].set_title(titles[i] if titles else f'Image {i+1}', fontsize=10)
        axes[0, i].axis('off')

        axes[1, i].imshow(overlay.clip(0, 1))
        axes[1, i].set_title(f'Pred: {LABEL_NAMES[pred_class]}\n{USCIS_LABELS[pred_class]}',
                             fontsize=9, color='red' if pred_class == 3 else 'black')
        axes[1, i].axis('off')

    plt.suptitle('GradCAM — TB Severity Classification\n(Red = high attention regions driving prediction)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('gradcam_results.png', dpi=150, bbox_inches='tight')
    plt.show()


print('GradCAM ready.')
print('Usage: visualize_gradcam(dn121_model, [img_path1, img_path2, img_path3, img_path4])')

## 9. Uncertainty Quantification (Monte Carlo Dropout)
Low-confidence predictions flagged for mandatory civil surgeon review.

In [ ]:
def enable_mc_dropout(model):
    """Keep dropout active during inference for MC sampling."""
    for module in model.modules():
        if isinstance(module, nn.Dropout):
            module.train()


def mc_predict(model, img_tensor, n_samples=30, device=DEVICE):
    """
    Monte Carlo Dropout prediction.
    Returns: mean prediction, uncertainty (entropy), confidence
    """
    model.eval()
    enable_mc_dropout(model)

    img_tensor = img_tensor.unsqueeze(0).to(device)
    probs_list = []

    with torch.no_grad():
        for _ in range(n_samples):
            logits = model(img_tensor)
            probs  = F.softmax(logits, dim=1)
            probs_list.append(probs.cpu().numpy())

    probs_array = np.stack(probs_list, axis=0).squeeze(1)  # (n_samples, 4)
    mean_probs  = probs_array.mean(axis=0)
    pred_class  = mean_probs.argmax()
    confidence  = mean_probs.max()

    # Predictive entropy (uncertainty)
    entropy = -np.sum(mean_probs * np.log(mean_probs + 1e-8))

    return {
        'prediction': pred_class,
        'label': LABEL_NAMES[pred_class],
        'uscis': USCIS_LABELS[pred_class],
        'confidence': float(confidence),
        'uncertainty': float(entropy),
        'flag_for_review': confidence < 0.75 or pred_class >= 2,
        'mean_probs': mean_probs
    }


CONFIDENCE_THRESHOLD = 0.75  # Below this → mandatory civil surgeon review

print('Uncertainty quantification ready.')
print(f'Confidence threshold for civil surgeon review: {CONFIDENCE_THRESHOLD}')
print('All Active TB and Severe TB predictions also flagged regardless of confidence.')

## 10. Demographic Equity Analysis (Cross-Demographic CTEI)
Extends MiST's CTEI from language groups to demographic groups.

In [ ]:
def equity_analysis(model, test_df, device, group_col='age_group'):
    """
    Compute per-demographic-group F1 and CTEI.
    Requires test_df to have a demographic column (age_group, sex, etc.)
    """
    if group_col not in test_df.columns:
        print(f'Column {group_col} not found. Add demographic metadata to test_df.')
        return

    results = {}
    for group in test_df[group_col].unique():
        sub_df = test_df[test_df[group_col] == group].reset_index(drop=True)
        ldr    = DataLoader(TBDataset(sub_df, val_transforms),
                            batch_size=BATCH_SIZE, shuffle=False)
        res = evaluate_per_class(model, ldr, device)
        results[group] = res['macro_f1']

    ctei = compute_ctei(results)

    print(f'\nCross-Demographic Triage Equity Index (CTEI): {ctei:.4f}')
    print(f'(Threshold for equitable deployment: CTEI >= 0.95)')
    print(f'{"Group":<20} {"Macro F1":>10}')
    print('-' * 32)
    for group, f1 in sorted(results.items(), key=lambda x: x[1]):
        print(f'{str(group):<20} {f1:>10.4f}')

    # Bar chart
    fig, ax = plt.subplots(figsize=(8, 4))
    groups = list(results.keys())
    f1s    = [results[g] for g in groups]
    colors = ['#2980B9' if f >= 0.85 else '#E74C3C' for f in f1s]
    ax.bar(groups, f1s, color=colors, alpha=0.85)
    ax.axhline(0.85, color='red', linestyle='--', label='Clinical threshold')
    ax.set(ylabel='Macro F1', title=f'Per-Group F1 (CTEI = {ctei:.3f})')
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('equity_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    return results, ctei


print('Equity analysis ready.')

## 11. Results Summary & Paper Tables

In [ ]:
def print_paper_tables(dn_results, rn_results, dn_ext=None, rn_ext=None):
    print('\nTable 2. Per-Class Performance — DenseNet-121 vs ResNet-50')
    print('=' * 75)
    header = f'{"Class":<20} {"DenseNet F1":>12} {"ResNet F1":>12} {"USCIS Outcome"}'
    print(header); print('-' * 75)
    for i, (name, uscis) in enumerate(zip(LABEL_NAMES, USCIS_LABELS)):
        dn_f1 = dn_results['per_class_f1'][i]
        rn_f1 = rn_results['per_class_f1'][i]
        print(f'{name:<20} {dn_f1:>12.4f} {rn_f1:>12.4f}  {uscis}')
    print('-' * 75)
    print(f'{"Macro F1":<20} {dn_results["macro_f1"]:>12.4f} {rn_results["macro_f1"]:>12.4f}')
    print(f'{"Severe TB Recall":<20} {dn_results["severe_recall"]:>12.4f} {rn_results["severe_recall"]:>12.4f}')

    if dn_ext and rn_ext:
        print('\nTable 3. Cross-Dataset Generalization (External Validation)')
        print('=' * 55)
        print(f'{"Dataset":<20} {"DenseNet F1":>12} {"ResNet F1":>12}')
        print('-' * 55)
        print(f'{"Montgomery":<20} {dn_ext.get("Montgomery",0):>12.4f} {rn_ext.get("Montgomery",0):>12.4f}')
        print(f'{"Shenzhen":<20} {dn_ext.get("Shenzhen",0):>12.4f} {rn_ext.get("Shenzhen",0):>12.4f}')


print('Results summary ready.')
print('Call: print_paper_tables(dn_results, rn_results) after evaluation.')

## Summary

| Component | Detail |
|---|---|
| **Task** | 4-class TB severity triage aligned with CDC/USCIS immigration medical exam |
| **Models** | DenseNet-121 (primary) + ResNet-50 (baseline) |
| **Transfer** | ImageNet → chest X-ray domain adaptation |
| **Loss** | Asymmetric Triage Loss (Severe TB weighted 5×) |
| **Datasets** | TBX11K (train) + Montgomery + Shenzhen (external validation) |
| **Explainability** | GradCAM — highlights lung regions driving classification |
| **Uncertainty** | Monte Carlo Dropout — flags ambiguous cases for civil surgeon review |
| **Fairness** | Cross-Demographic CTEI (adapted from MiST) |
| **NIW Hook** | Directly assists USCIS Form I-693 immigration medical examination |

### 4-Class USCIS Mapping
| CNN Output | Clinical Label | USCIS Outcome |
|---|---|---|
| 0 | Normal | Cleared |
| 1 | Inactive TB | Class B2 — cleared with 30-day follow-up |
| 2 | Active TB | Class B1 — sputum test + evaluation required |
| 3 | Severe TB | Class A — immigration denied until treatment complete |

### Outputs
- `training_curves.png`
- `gradcam_results.png`
- `equity_analysis.png`
- `checkpoints/densenet121/best.pt`
- `checkpoints/resnet50/best.pt`